# Predict `rate` for 10 preprocessed reviews

This notebook scores 10 rows with `experiments/baselines/tookabert-epoch2-regression`. The encoder is TookaBERT preprocessing-v2 epoch 2. The one-output layer is new and has not been trained on `rate`, so the numbers are a forward pass, not a fitted rating model.

The rows come from `data/processed/digikala-comments-preprocessed.csv`. Each body then gets the epoch-2 stages: letters, digits, emoji. The ParsBERT token parquet is not used. Run this notebook from the repository root, or from this folder.


In [3]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(ROOT)
CSV_PATH = ROOT / "data/processed/digikala-comments-preprocessed.csv"
MODEL_DIR = ROOT / "experiments/baselines/tookabert-epoch2-regression"
N_ROWS = 10
MAX_LENGTH = 128

if not CSV_PATH.is_file():
    raise FileNotFoundError(CSV_PATH)
if not (MODEL_DIR / "model.safetensors").is_file():
    raise FileNotFoundError(MODEL_DIR)
print(ROOT)


/Users/amir/Documents/GitHub/digikala-products-recommender-systems
/Users/amir/Documents/GitHub/digikala-products-recommender-systems


## Ten rows

The read stops after 200 rows. The first 10 bodies that are still non-empty after the three text stages are the sample.


In [4]:
import polars as pl

from src.experiments.text_v2 import apply_digits, apply_emoji, apply_letters


def prepare_body(text: str) -> str:
    return apply_emoji(apply_digits(apply_letters(text or "")))


frame = pl.read_csv(
    CSV_PATH,
    n_rows=200,
    columns=["id", "body", "rate"],
    schema_overrides={"id": pl.Int64, "body": pl.String, "rate": pl.Float64},
)
rewritten = [prepare_body(text) for text in frame.get_column("body").to_list()]
sample = (
    frame.with_columns(pl.Series("body", rewritten))
    .filter(pl.col("body") != "")
    .head(N_ROWS)
)
if sample.height != N_ROWS:
    raise RuntimeError(f"needed {N_ROWS} non-empty bodies, found {sample.height}")
sample.select("id", "rate", "body")


id,rate,body
i64,f64,str
53672599,1.0,"""به درد نمیخوره"""
9897229,0.0,"""می‌تونس به عنو یه کالا فرهنگ ب…"
38074516,0.0,"""بسته بند خوب بود کاربرد و کیفی…"
18628562,0.0,"""به نظر خوبه فقط یک ظریفه . از …"
53301258,3.0,"""معمولیه اگه واسه خونه رنگ کردن…"
53266157,5.0,"""قبلا ه استفاده کرد اگه بلد باش…"
40597171,5.0,"""خیل به کارتون میاد قیمت مناسبه"""
46198656,5.0,"""دقیقا مطابق عکس بود"""
46901811,5.0,"""برا حال دادن با اسپر یا پودر ع…"


## Predict

The tokenizer and the weights are the regression baseline. `predicted_raw` is the single model output. `predicted_rate` is that number clipped to [0, 5].


In [5]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

encoded = tokenizer(
    sample.get_column("body").to_list(),
    truncation=True,
    max_length=MAX_LENGTH,
    padding=True,
    return_tensors="pt",
)
with torch.inference_mode():
    raw = model(**encoded).logits.squeeze(-1).tolist()
clipped = [min(5.0, max(0.0, float(value))) for value in raw]
result = sample.select("id", "rate").with_columns(
    pl.Series("predicted_raw", [float(value) for value in raw]),
    pl.Series("predicted_rate", clipped),
)
result


W1118 14:28:14.944000 70479 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

id,rate,predicted_raw,predicted_rate
i64,f64,f64,f64
53672599,1.0,0.225172,0.225172
9897229,0.0,0.248961,0.248961
38074516,0.0,0.05896,0.05896
18628562,0.0,0.149381,0.149381
53301258,3.0,0.299538,0.299538
53266157,5.0,0.037211,0.037211
40597171,5.0,0.059446,0.059446
46198656,5.0,0.302658,0.302658
46901811,5.0,0.098624,0.098624


In [6]:
rate_counts = (
    pl.scan_parquet(ROOT / "data/tokenized/digikala-comments-tokenized.parquet")
    .group_by("rate")
    .agg(pl.len().alias("count"))
    .sort("rate")
    .collect()
)
print("preprocessed", rate_counts.height, "distinct rates,", int(rate_counts["count"].sum()), "rows")
display(rate_counts)

NA_VALUES = ["", "nan", "NaN", "NAN", "null", "NULL", "None", "none", "-"]
raw_counts = (
    pl.scan_csv(
        ROOT / "data/raw/digikala-comments.csv",
        null_values=NA_VALUES,
    )
    .select(pl.col("rate").cast(pl.Float64, strict=False))
    .group_by("rate")
    .agg(pl.len().alias("count"))
    .sort("rate")
    .collect()
)
print("raw", raw_counts.height, "distinct rates,", int(raw_counts["count"].sum()), "rows")
raw_counts


preprocessed 171 distinct rates, 5258342 rows


rate,count
f64,u32
0.0,503570
0.05,94
0.1,62
0.15,173
0.2,570
…,…
4.8,6267
4.85,1292
4.88,2


raw 178 distinct rates, 6156289 rows


rate,count
f64,u32
0.0,529098
0.05,97
0.1,66
0.15,181
0.2,661
…,…
4.85,1476
4.88,2
4.9,426
